In [1]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
import re
from functools import reduce
from operator import add

In [2]:
from pyspark.sql import SparkSession
from src.utils.logger import get_logger
import src.utils.config as config 


print(f"DEBUG: Access Key is {config.MINIO_ACCESS_KEY[:3]} + ***") 
print(f"DEBUG: ENDPOINT is {config.MINIO_ENDPOINT}")

# Get the container's hostname dynamically

logger = get_logger(__name__)

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a configured Spark session for MinIO.
    """
    logger.info(f"Creating Spark Session: {app_name}")
    
    # This pulls the necessary S3A connectors from Maven Central
    
    

    spark = (   
        SparkSession.builder
        .appName(app_name)
        .master("spark://spark-master:7077")
        .config("spark.driver.host", config.DRIVER_HOST)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.port", config.SPARK_DRIVER_PORT)
        .config("spark.driver.blockManager.port", config.SPARK_BLOCK_MANAGER_PORT)
        .config("spark.sql.shuffle.partitions", "50")
        .config("spark.executor.instances", "1") # adjust based on resources
        .config("spark.executor.cores", "2")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
        #.config("spark.executor.memory", "2g") # adjust based on resources
        #.config("spark.driver.memory", "2g") # adjust based on resources

         # hadoop S3A Configuration
      
        .config("spark.hadoop.fs.s3a.endpoint", config.MINIO_ENDPOINT)
        .config("spark.hadoop.fs.s3a.access.key", config.MINIO_ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", config.MINIO_SECRET_KEY)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(config.MINIO_SECURE).lower())
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # prevent class resolution
        .config("spark.sql.caseSensitive", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.cores.max", "2")
        .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.hadoop.fs.s3a.fast.upload", "true") #performance improvement
        .config("spark.sql.files.maxPartitionBytes", "16777216") #16MB partitions
        #.config("spark.network.timeout", "1200s")
        #.config("spark.rpc.askTimeout", "600s")
        #.config("spark.executor.heartbeatInterval", "120s")
        #.config("spark.hadoop.fs.s3a.connection.timeout", "600000")
        #.config("spark.hadoop.fs.s3a.paging.maximum", "1000")
        
        .getOrCreate()
    )
    # Suppress verbose logs
    spark.sparkContext.setLogLevel("WARN")
    logger.info("Spark session created successfully")
    return spark

[CONFIG] Stage: transform
[CONFIG] Loaded env: /opt/spark-app/env/.env.transform
[CONFIG] MinIO Endpoint: lakehouse-minio:9000
[CONFIG] Access Key (masked): tra***
DEBUG: Access Key is tra + ***
DEBUG: ENDPOINT is lakehouse-minio:9000


In [3]:
spark= create_spark_session("notebook_test")
ireland_2024= spark.read.parquet("s3a://bronze/IRELAND/2024_BRONZE/")

2026-07-25 07:59:57 | INFO | lakehouse.__main__ | Creating Spark Session: notebook_test


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/25 08:03:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-07-25 08:04:01 | INFO | lakehouse.__main__ | Spark session created successfully


26/07/25 08:04:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
ireland_24=ireland_2024
ireland_24

DataFrame[information_for_the_purposes_of_transparency_pursuant_to_article_58: string, unnamed:_1: string, unnamed:_2: string, unnamed:_3: string, unnamed:_4: string, unnamed:_5: string, unnamed:_6: string, unnamed:_7: string, unnamed:_8: string, unnamed:_9: string, unnamed:_10: string, unnamed:_11: string, unnamed:_12: string, unnamed:_13: string, unnamed:_14: string, unnamed:_15: string, source_country: string, source_year: string, ingested_at: timestamp]

In [5]:
ireland_24.rdd.getNumPartitions()

2

In [5]:
ireland_24.printSchema()

root
 |-- information_for_the_purposes_of_transparency_pursuant_to_article_58: string (nullable = true)
 |-- unnamed:_1: string (nullable = true)
 |-- unnamed:_2: string (nullable = true)
 |-- unnamed:_3: string (nullable = true)
 |-- unnamed:_4: string (nullable = true)
 |-- unnamed:_5: string (nullable = true)
 |-- unnamed:_6: string (nullable = true)
 |-- unnamed:_7: string (nullable = true)
 |-- unnamed:_8: string (nullable = true)
 |-- unnamed:_9: string (nullable = true)
 |-- unnamed:_10: string (nullable = true)
 |-- unnamed:_11: string (nullable = true)
 |-- unnamed:_12: string (nullable = true)
 |-- unnamed:_13: string (nullable = true)
 |-- unnamed:_14: string (nullable = true)
 |-- unnamed:_15: string (nullable = true)
 |-- source_country: string (nullable = true)
 |-- source_year: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [6]:
for i , col in enumerate(ireland_24.columns, 1):
    print(f"{i:3d}| {col}")

  1| information_for_the_purposes_of_transparency_pursuant_to_article_58
  2| unnamed:_1
  3| unnamed:_2
  4| unnamed:_3
  5| unnamed:_4
  6| unnamed:_5
  7| unnamed:_6
  8| unnamed:_7
  9| unnamed:_8
 10| unnamed:_9
 11| unnamed:_10
 12| unnamed:_11
 13| unnamed:_12
 14| unnamed:_13
 15| unnamed:_14
 16| unnamed:_15
 17| source_country
 18| source_year
 19| ingested_at


In [7]:
ireland_24.select(
    "information_for_the_purposes_of_transparency_pursuant_to_article_58",
    "unnamed:_1",
    "unnamed:_2",
    "unnamed:_3"
    
).limit(10).toPandas()


,information_for_the_purposes_of_transparency_pursuant_to_article_58,unnamed:_1,unnamed:_2,unnamed:_3
0,None,None,None,None
1,Name of the beneficiary/Legal entity/association,Surname of beneficiary,"If belonging to a group, name of the parent en...",Municipality
2,None,None,None,None
3,MICHAEL CLANCY,CLANCY,None,LONGFORD
4,MICHAEL CLANCY,CLANCY,None,LONGFORD
5,MICHAEL CLANCY,CLANCY,None,LONGFORD
6,MICHAEL CLANCY,CLANCY,None,LONGFORD
7,MICHAEL CLANCY,CLANCY,None,LONGFORD
8,MICHAEL CLANCY,CLANCY,None,LONGFORD
9,None,None,None,None


In [6]:
#since we know that the column name is in the second row , we are extracting it and creating a new dataframe
new_column_names= []

sample_rows= ireland_24.take(2)[1]

legit_columns= ["source_country", "source_year", "ingested_at"]
# Conveert the raw values into a clean python list

for i , old_name in enumerate(ireland_24.columns):
    if old_name in legit_columns:
        new_column_names.append(old_name)
    else:
        new_column_names.append(str(sample_rows[i]))
# map the old column names to the new one
rename_map= dict(zip(ireland_24.columns, new_column_names))
df_with_headers= ireland_24.withColumnsRenamed(rename_map)

sample_new_col= [col for col in new_column_names if col not in legit_columns][0]
ireland_24_new=df_with_headers.filter(F.col(sample_new_col) != sample_new_col) 
ireland_24_new.show(5, truncate=False)


+------------------------------------------------+----------------------+---------------------------------------------------------------------------------------+------------+--------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+-------------------+------------------------------+-----------------------------------------+-------------------------------+------------------------------------------+--------------------------------------+------------------------------------------------+--------------------------------------+-------------------------------------------+--------------+------

In [16]:
ireland_24_new.show(5)

+------------------------------------------------+----------------------+---------------------------------------------------------------------------------------+------------+--------------------------------------------------------------+---------------------+-------------------+-------------------+------------------------------+-----------------------------------------+-------------------------------+------------------------------------------+--------------------------------------+------------------------------------------------+--------------------------------------+-------------------------------------------+--------------+-----------+--------------------+
|Name of the beneficiary/Legal entity/association|Surname of beneficiary|If belonging to a group, name of the parent entity and VAT or Tax identification number|Municipality|Code of measure type of intervention/sector as set in Annex IX|Specific objective Â¹|      Start date Â²|        End date Â³|Amount by operation under EAGF|Tot

In [7]:
# let's know the number of non null or nan values present in each column

# 1. Build expressions dynamically based on each column's specific data type
count_expressions = []


for col_name, col_type in ireland_24_new.dtypes:
    # Base condition: works for ALL data types (Timestamps, Strings, Ints, etc.)
    condition = F.col(col_name).isNotNull()

    # Only append the NaN check if the column is a floating-point numeric type
    if col_type in ("double", "float"):
        condition = condition & ~F.isnan(F.col(col_name))
    
    # Rule B: Catch empty text cells in string columns
    elif col_type == "string":
        condition = condition & ~(F.trim(F.col(col_name))).isin("", "N/A", "n/a", "NA", "na")
    
    # Aggregate using the safe conditional block
    count_expressions.append(F.count(F.when(condition, 1)).alias(col_name))

# 2. Run the single, optimized aggregation across the cluster
counts_row = ireland_24_new.select(count_expressions).first()

print(counts_row)


Row(Name of the beneficiary/Legal entity/association=571003, Surname of beneficiary=480559, If belonging to a group, name of the parent entity and VAT or Tax identification number=4216, Municipality=570172, Code of measure type of intervention/sector as set in Annex IX=570172, Specific objective Â¹=544317, Start date Â²=544179, End date Â³=544248, Amount by operation under EAGF=377289, Total of EAGF amount for that beneficiary=0, Amount by operation under EAFRD=192814, Total of EAFRD amount for that beneficiary=0, Amount by operation under co-financing=178964, Total of co-financed amount for that beneficiary=0, Total of EAFRD and co-financed amounts=0, Total of the EU amount for that beneficiary=0, source_country=571003, source_year=571003, ingested_at=571003)


In [8]:
# Count total rows in the DataFrame
total_rows = ireland_24_new.count()

# Print header
print(f'{"Column Name": <65} | {"Missing Percentage"}')
print("-" * 85)

# Calculate and print missing percentage for each column
for column, valid_count in counts_row.asDict().items():
    missing_percentage = ((total_rows - valid_count) / total_rows) * 100
    print(f"{column: <65} | {missing_percentage: >10.3f}%")


Column Name                                                       | Missing Percentage
-------------------------------------------------------------------------------------
Name of the beneficiary/Legal entity/association                  |      0.000%
Surname of beneficiary                                            |     15.839%
If belonging to a group, name of the parent entity and VAT or Tax identification number |     99.262%
Municipality                                                      |      0.146%
Code of measure type of intervention/sector as set in Annex IX    |      0.146%
Specific objective Â¹                                             |      4.674%
Start date Â²                                                     |      4.698%
End date Â³                                                       |      4.686%
Amount by operation under EAGF                                    |     33.925%
Total of EAGF amount for that beneficiary                         |    100.000%
Amoun

In [8]:
# Get column data types as a dictionary
col_types = dict(ireland_24_new.dtypes)

# Count total rows in the DataFrame
total_rows = ireland_24_new.count()

# Initialize list to hold stats
stats_list = []

# Extract stats from data
for col_name, valid_count in counts_row.asDict().items():
    missing_count = total_rows - valid_count
    missing_percentage = round((missing_count / total_rows) * 100, 2)

    stats_list.append({
        'column_name': col_name,
        'missing_count': missing_count,
        'missing_percentage': missing_percentage,
        'Non-Null count': valid_count,
        'dtype': col_types.get(col_name)
    })
# Sort missing_percentage in ascending order
missing_ireland_stats_2024 = sorted(
    stats_list,
    key=lambda x: x['missing_percentage']
)

# Display the sorted list
missing_ireland_stats_2024

# Convert to DataFrame (optional, for display)
import pandas as pd
stats_df = pd.DataFrame(stats_list)

# Show nicely formatted table
print(stats_df)




                                          column_name  missing_count  \
0    Name of the beneficiary/Legal entity/association              0   
1                              Surname of beneficiary          90444   
2   If belonging to a group, name of the parent en...         566787   
3                                        Municipality            831   
4   Code of measure type of intervention/sector as...            831   
5                               Specific objective Â¹          26686   
6                                       Start date Â²          26824   
7                                         End date Â³          26755   
8                      Amount by operation under EAGF         193714   
9           Total of EAGF amount for that beneficiary         571003   
10                    Amount by operation under EAFRD         378189   
11         Total of EAFRD amount for that beneficiary         571003   
12             Amount by operation under co-financing         39

In [9]:
ireland_24_cleaned= ireland_24_new.select(
    F.col("Name of the beneficiary/Legal entity/association").alias("beneficiary"),
    F.col("Municipality").alias("municipality"),
    F.col("source_country").alias("country"),
    F.col("source_year").alias("year"),
    
    F.col("Code of measure type of intervention/sector as set in Annex IX").alias("intervention_code"),
    F.col("Amount by operation under EAGF").cast(DoubleType()).alias("total_eagf_income_support"),
    F.col("Amount by operation under EAFRD").cast(DoubleType()).alias("total_eafrd_income_support"),
    F.col("Amount by operation under co-financing").cast(DoubleType()).alias("national_cofunding_amount")
).fillna(0.0, subset=["total_eagf_income_support", "total_eafrd_income_support", "national_cofunding_amount"])

# fill nulls in text fields
ireland24_cleaned= ireland_24_cleaned.fillna("UNKNOWN", subset= ["beneficiary", "municipality", "intervention_code"])

In [11]:
ireland24_cleaned.show(5, truncate=False)

+--------------+------------+-------+----+-----------------+-------------------------+--------------------------+-------------------------+
|beneficiary   |municipality|country|year|intervention_code|total_eagf_income_support|total_eafrd_income_support|national_cofunding_amount|
+--------------+------------+-------+----+-----------------+-------------------------+--------------------------+-------------------------+
|MICHAEL CLANCY|LONGFORD    |IRELAND|2024|V.4              |0.0                      |263.09                    |176.94                   |
|MICHAEL CLANCY|LONGFORD    |IRELAND|2024|I.1              |13716.37                 |0.0                       |0.0                      |
|MICHAEL CLANCY|LONGFORD    |IRELAND|2024|I.4              |9895.23                  |0.0                       |0.0                      |
|MICHAEL CLANCY|LONGFORD    |IRELAND|2024|I.2              |1339.5                   |0.0                       |0.0                      |
|MICHAEL CLANCY|LONG

In [12]:
spark.stop()